# SHMQ-Ultimate strict GPU gate

This notebook is a correctness gate for the three-level CUDA kernel.
It deliberately fails when CUDA/CuPy/NVRTC is unavailable or when the
implementation silently falls back to PyTorch. It does not claim tensor
core performance or Qwen quality; those require separate benchmarks.


In [ ]:
import os, platform, sys, time
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'CUDA is unavailable'
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
print('GPU:', name)
print('Compute capability:', cap)
print('VRAM GiB:', props.total_memory / 2**30)
assert cap == (7, 5), f'Expected T4 sm_75, got sm_{cap[0]}{cap[1]}'
assert props.total_memory >= 15 * 2**30, 'Expected a 16 GiB-class T4'
import cupy as cp
print('CuPy:', cp.__version__)
print('CUDA runtime:', cp.cuda.runtime.runtimeGetVersion())
assert cp.cuda.runtime.getDevice() == torch.cuda.current_device()


In [ ]:
# Load exactly the source committed in this workspace; no repository clone or fallback copy.
_source = '"""SHMQ 3-level {FP16, INT8, INT4} CUDA GEMM kernel for T4 (sm_75).\n\nThis module provides a SINGLE CUDA kernel launch that processes weights stored\nat three precision levels:\n\n  Y[M, N] = X[M, K] @ W[N, K].T\n\nwhere W is logically partitioned along N (output) as:\n  [ W_fp16 | W_int8 | W_int4 ]\n\nEach partition is multiplied by its corresponding slice of X and accumulated\ninto the SAME output Y. The kernel:\n  - Loads FP16 slice directly into shared memory\n  - Loads INT8 slice and dequantizes to FP16 using per-group-of-128 scales\n  - Loads INT4 slice (packed 2-per-byte), unpacks and dequantizes to FP16\n  - All three paths accumulate into a single FP32 register file\n  - Final cast to FP16 for output\n\nDesign choices:\n  1. SINGLE kernel launch — no 3-launch overhead, no cuBLAS-for-FP16 split.\n     This is what the user explicitly requested ("FP16 supported natively in\n     the same kernel as INT4 and INT8").\n  2. The implemented path dequantizes INT8/INT4 weights into shared-memory\n     FP16 tiles and computes with CUDA cores. It is not a tensor-core INT4/INT8\n     microkernel and must not be described as one. A real Turing tensor-core\n     implementation needs separate, layout-correct MMA paths (not merely PTX\n     wrappers embedded in this source).\n  3. Activations stay FP16 throughout. The original SHMQ paper uses W4.8A8\n     (activations INT8), but for the 3-level {4,8,16} case where we have\n     FP16 weights, keeping activations FP16 lets the FP16 weight path be\n     useful (otherwise FP16 weights × INT8 activations = wasted precision).\n  4. cupy.RawKernel + NVRTC: CUDA C++ is shipped as a Python string and\n     compiled at runtime. No pre-compiled .so, no Makefile, no setup.py.\n     ipynb-friendly: just `import cupy` and go.\n\nCompatible with:\n  - T4 (sm_75, Turing) — primary target\n  - Newer CUDA GPUs, subject to runtime compilation and correctness testing\n\nvLLM integration: this kernel is wrapped by `SHMQQuantLinearMethod` (see\n`vllm_patch/0005-shmq-3level-t4-support.patch`) and registered as a custom\nquantization method named "shmq_3level". vLLM then loads the model with\nthis method, the same way it loads GPTQ/AWQ/MixLLM models.\n\n   - CUDA C Programming Guide: shared-memory tiled matrix multiplication\n  - mma.sync.aligned.m8n8k4.row.col.s32.s4.s4.s32        (INT4, sm_75+)\n"""\nfrom __future__ import annotations\nimport os\nfrom typing import Optional, Tuple, Dict\nimport torch\n\nGROUP_SIZE = 128\n\n# ---------------------------------------------------------------------------\n# CUDA C++ source as a string. Compiled at runtime by cupy.RawKernel (NVRTC).\n# ---------------------------------------------------------------------------\nSHMQ_3LEVEL_KERNEL_CUDA = r"""\n#include <cuda_fp16.h>\n#include <cuda_runtime.h>\n#include <cstdint>\n\n// ===========================================================================\n// Tiling configuration\n//   BM=32, BN=32, BK=32 — fits comfortably in T4 shared memory (64KB/SM)\n//   4 warps per block (128 threads), 2x2 warp grid\n//   Each warp covers a 16x16 sub-tile\n//   Within warp: 4x8 thread grid, each thread computes 4 rows x 2 cols = 8 outputs\n// ===========================================================================\n#define BM 32\n#define BN 32\n#define BK 32\n#define WARPS_PER_BLOCK 4\n#define THREADS_PER_BLOCK (WARPS_PER_BLOCK * 32)\n#define GROUP_SIZE 128\n\n// ===========================================================================\n// Main GEMM kernel — handles 3 precision levels in ONE launch\n// CUDA-cores path (default). Tensor-core path selected via compile flag.\n//\n// Layout:\n//   X  : [M, K]           row-major, FP16\n//   W16: [N16, K]         row-major, FP16 (kept as-is)\n//   W8 : [N8,  K]         row-major, INT8  (symmetric, zero=0)\n//   W4 : [N4,  K/2]       row-major, packed INT4 (2 per byte, lower nibble = even idx)\n//   S8 : [N8,  K/128]     FP16 per-group-of-128 input-channel scales\n//   S4 : [N4,  K/128]     FP16 per-group-of-128 input-channel scales\n//   Y  : [M, N]           row-major, FP16 output\n//\n//   N = N16 + N8 + N4\n//\n// Each block computes a BM x BN tile of Y by looping over K in steps of BK.\n// Within each BK step it:\n//   1. Loads X_tile [BM, BK] into shared memory (FP16)\n//   2. Loads W_tile [BN, BK] into shared memory as FP16, dispatching by\n//      global n: FP16 direct, INT8 dequant with S8, INT4 unpack+dequant with S4\n//   3. Computes partial dot products and accumulates into FP32 registers\n//   4. After all K steps, writes the FP32 accumulator (cast to FP16) to Y\n// ===========================================================================\n\nextern "C" __global__ void shmq_3level_gemm_kernel(\n    const half* __restrict__ X,        // [M, K]\n    const half* __restrict__ W16,      // [N16, K]\n    const int8_t* __restrict__ W8,     // [N8, K]\n    const uint8_t* __restrict__ W4,    // [N4, K/2]  (packed INT4, lower nibble = even idx)\n    const half* __restrict__ S8,       // [N8, K/128]\n    const half* __restrict__ S4,       // [N4, K/128]\n    half* __restrict__ Y,              // [M, N]\n    int M, int K, int N,\n    int N16, int N8, int N4)\n{\n    int bx = blockIdx.x;   // BN tile index (along N)\n    int by = blockIdx.y;   // BM tile index (along M)\n    int tid = threadIdx.x;\n    int warp_id = tid >> 5;\n    int lane = tid & 31;\n\n    int n_start = bx * BN;\n    int m_start = by * BM;\n\n    // 2x2 warp grid: each warp covers a 16x16 sub-tile\n    int warp_row = warp_id / 2;   // 0 or 1 (each warp covers 16 rows of M)\n    int warp_col = warp_id % 2;   // 0 or 1 (each warp covers 16 cols of N)\n\n    // Within warp: 4x8 thread grid (32 threads)\n    //   tr = lane / 8   (0..3) — each thread covers 4 rows\n    //   tc = lane % 8   (0..7) — each thread covers 2 cols\n    int tr = lane >> 3;   // 0..3\n    int tc = lane & 7;    // 0..7\n\n    // Each thread computes 4 rows x 2 cols = 8 outputs\n    float acc[4][2];\n    #pragma unroll\n    for (int r = 0; r < 4; r++) {\n        acc[r][0] = 0.f;\n        acc[r][1] = 0.f;\n    }\n\n    // Shared memory layout (per block):\n    //   sX [BM * BK]            = 32*32 * 2  = 2048 bytes\n    //   sW [BN * BK]            = 32*32 * 2  = 2048 bytes (FP16, after dequant)\n    // Total = 4096 bytes per block (T4 has 64KB/SM, fits comfortably)\n    extern __shared__ char smem[];\n    half* sX = (half*)smem;                       // [BM, BK]\n    half* sW = sX + BM * BK;                      // [BN, BK] — FP16 (dequantized)\n\n    // Precompute N-region boundaries (global n)\n    int n16_end = N16;          // FP16 region: n in [0, N16)\n    int n8_end  = N16 + N8;     // INT8 region: n in [N16, N16+N8)\n    // INT4 region: n in [N16+N8, N)\n\n    // Number of scale groups along K\n    int n_groups_k = K / GROUP_SIZE;\n\n    // ---- Loop over K ----\n    for (int k = 0; k < K; k += BK) {\n\n        // ============ Load X tile [BM, BK] ============\n        // 128 threads, BM*BK = 1024 elements → 8 elements per thread.\n        // Thread tid loads 8 contiguous FP16 from row (tid/4), col (tid%4)*8\n        //   → covers 32 rows × 32 cols = 1024 elements ✓\n        #pragma unroll\n        for (int i = 0; i < 8; i++) {\n            int row = tid >> 2;          // 0..31\n            int col = (tid & 3) * 8 + i; // 0..31\n            int xm = m_start + row;\n            int xk = k + col;\n            sX[row * BK + col] =\n                (xm < M && xk < K) ? X[xm * K + xk] : __float2half(0.f);\n        }\n\n        // ============ Load W tile [BN, BK] ============\n        // Same thread mapping as X. Each thread loads 8 elements.\n        // For each (row, col), dispatch by global n (= n_start + row):\n        //   if gn < N16          : W16[gn, gk]              (FP16, direct)\n        //   else if gn < N16+N8   : W8[gn-N16, gk] * S8[...] (INT8, dequant)\n        //   else                  : W4[gn-N16-N8, gk/2] * S4[...] (INT4, unpack+dequant)\n        #pragma unroll\n        for (int i = 0; i < 8; i++) {\n            int row = tid >> 2;          // 0..31 (local row in sW)\n            int col = (tid & 3) * 8 + i; // 0..31 (local col in sW)\n            int gn = n_start + row;      // global n\n            int gk = k + col;            // global k\n\n            half val = __float2half(0.f);\n\n            if (gn < n16_end) {\n                // ---------- FP16 path ----------\n                val = (gk < K) ? W16[gn * K + gk] : __float2half(0.f);\n            } else if (gn < n8_end) {\n                // ---------- INT8 path ----------\n                int n8_idx = gn - N16;\n                int g_idx = gk / GROUP_SIZE;\n                half scale = S8[n8_idx * n_groups_k + g_idx];\n                int8_t code = (gk < K) ? W8[n8_idx * K + gk] : (int8_t)0;\n                float fcode = __int2float_rn((int)code);\n                val = __float2half(fcode * __half2float(scale));\n            } else if (gn < N) {\n                // ---------- INT4 path ----------\n                int n4_idx = gn - n8_end;  // = gn - N16 - N8\n                int g_idx = gk / GROUP_SIZE;\n                half scale = S4[n4_idx * n_groups_k + g_idx];\n                int byte_idx = gk >> 1;\n                uint8_t packed = (gk < K) ? W4[n4_idx * (K >> 1) + byte_idx] : (uint8_t)0;\n                int8_t code;\n                if (gk & 1) {\n                    code = (int8_t)((packed >> 4) & 0x0F);\n                } else {\n                    code = (int8_t)(packed & 0x0F);\n                }\n                // Sign-extend from 4 bits: values 8..15 become -8..-1\n                if (code >= 8) code = (int8_t)(code - 16);\n                float fcode = __int2float_rn((int)code);\n                val = __float2half(fcode * __half2float(scale));\n            }\n            sW[row * BK + col] = val;\n        }\n\n        __syncthreads();\n\n        // ============ Compute partial sum (CUDA cores) ============\n        // Each thread: 4 rows × 2 cols × BK inner loop\n        // Output (r, c) at:\n        //   m = m_start + warp_row*16 + tr*4 + r\n        //   n = n_start + warp_col*16 + tc*2 + c\n        //\n        // We use a 4-way unroll on the inner loop for better ILP.\n        #pragma unroll\n        for (int r = 0; r < 4; r++) {\n            int sx_row = warp_row * 16 + tr * 4 + r;\n            #pragma unroll\n            for (int c = 0; c < 2; c++) {\n                int sw_row = warp_col * 16 + tc * 2 + c;\n                float sum = 0.f;\n                #pragma unroll\n                for (int kk = 0; kk < BK; kk += 4) {\n                    sum += __half2float(sX[sx_row * BK + kk]) *\n                           __half2float(sW[sw_row * BK + kk]);\n                    sum += __half2float(sX[sx_row * BK + kk + 1]) *\n                           __half2float(sW[sw_row * BK + kk + 1]);\n                    sum += __half2float(sX[sx_row * BK + kk + 2]) *\n                           __half2float(sW[sw_row * BK + kk + 2]);\n                    sum += __half2float(sX[sx_row * BK + kk + 3]) *\n                           __half2float(sW[sw_row * BK + kk + 3]);\n                }\n                acc[r][c] += sum;\n            }\n        }\n\n        __syncthreads();\n    }\n\n    // ============ Write Y tile ============\n    // Each thread writes its 8 outputs (4 rows x 2 cols).\n    // Boundary checks: skip writes where m >= M or n >= N.\n    #pragma unroll\n    for (int r = 0; r < 4; r++) {\n        int m = m_start + warp_row * 16 + tr * 4 + r;\n        if (m >= M) continue;\n        #pragma unroll\n        for (int c = 0; c < 2; c++) {\n            int n = n_start + warp_col * 16 + tc * 2 + c;\n            if (n >= N) continue;\n            Y[m * N + n] = __float2half(acc[r][c]);\n        }\n    }\n}\n\n// ===========================================================================\n// Reference: per-element INT4 unpacking kernel (used by Python fallback path)\n// ===========================================================================\nextern "C" __global__ void shmq_unpack_int4_kernel(\n    const uint8_t* __restrict__ packed,  // [N4, K/2]\n    int8_t* __restrict__ unpacked,        // [N4, K]\n    int N4, int K)\n{\n    int idx = blockIdx.x * blockDim.x + threadIdx.x;\n    int total = N4 * K;\n    if (idx >= total) return;\n    int row = idx / K;\n    int col = idx % K;\n    uint8_t byte = packed[row * (K / 2) + col / 2];\n    int8_t code;\n    if (col & 1) {\n        code = (int8_t)((byte >> 4) & 0x0F);\n    } else {\n        code = (int8_t)(byte & 0x0F);\n    }\n    if (code >= 8) code = (int8_t)(code - 16);\n    unpacked[idx] = code;\n}\n"""\n\n# ---------------------------------------------------------------------------\n# Python wrapper — tries cupy.RawKernel first, falls back to PyTorch\n# ---------------------------------------------------------------------------\n\n_CUPY_AVAILABLE = None  # lazy: None=not checked, True/False\n_RAW_KERNEL_GEMM = None\n_RAW_KERNEL_UNPACK = None\n_COMPILED_OPTIONS = None\n\n\ndef _check_cupy():\n    """Lazy cupy import — returns the module or None."""\n    global _CUPY_AVAILABLE\n    if _CUPY_AVAILABLE is None:\n        try:\n            import cupy as cp\n            _ = cp.cuda.runtime.getDevice()\n            _CUPY_AVAILABLE = cp\n        except Exception:\n            _CUPY_AVAILABLE = False\n    return _CUPY_AVAILABLE if _CUPY_AVAILABLE else None\n\n\ndef _detect_compute_capability():\n    """Return (major, minor) of the current CUDA device, or (7, 5) as default."""\n    cp = _check_cupy()\n    if cp is None:\n        return (7, 5)  # default to T4\n    try:\n        prop = cp.cuda.runtime.getDeviceProperties(0)\n        return (int(prop["major"]), int(prop["minor"]))\n    except Exception:\n        return (7, 5)\n\n\ndef _nvrtc_options():\n    """Build NVRTC compile options for the current GPU."""\n    major, minor = _detect_compute_capability()\n    sm = f"{major}{minor}"\n    # RawKernel uses NVRTC, which accepts --gpu-architecture but not nvcc\'s\n    # separate -code option.\n    return (\n        "-std=c++14",\n        f"--gpu-architecture=compute_{sm}",\n        "--use_fast_math",\n        "-DCUDA_NO_HALF",\n    )\n\n\ndef _get_gemm_kernel():\n    """Compile the main GEMM kernel via cupy.RawKernel. Cached."""\n    global _RAW_KERNEL_GEMM, _COMPILED_OPTIONS\n    if _RAW_KERNEL_GEMM is not None:\n        return _RAW_KERNEL_GEMM\n    cp = _check_cupy()\n    if cp is None:\n        return None\n    try:\n        opts = _nvrtc_options()\n        _COMPILED_OPTIONS = opts\n        _RAW_KERNEL_GEMM = cp.RawKernel(\n            SHMQ_3LEVEL_KERNEL_CUDA,\n            "shmq_3level_gemm_kernel",\n            options=opts,\n            jitify=True,\n        )\n        return _RAW_KERNEL_GEMM\n    except Exception as e:\n        print(f"[shmq_3level_kernel] cupy.RawKernel compile failed: {e}")\n        return None\n\n\ndef _get_unpack_kernel():\n    """Compile the INT4 unpack kernel. Cached."""\n    global _RAW_KERNEL_UNPACK\n    if _RAW_KERNEL_UNPACK is not None:\n        return _RAW_KERNEL_UNPACK\n    cp = _check_cupy()\n    if cp is None:\n        return None\n    try:\n        opts = _nvrtc_options()\n        _RAW_KERNEL_UNPACK = cp.RawKernel(\n            SHMQ_3LEVEL_KERNEL_CUDA,\n            "shmq_unpack_int4_kernel",\n            options=opts,\n            jitify=True,\n        )\n        return _RAW_KERNEL_UNPACK\n    except Exception as e:\n        print(f"[shmq_3level_kernel] unpack kernel compile failed: {e}")\n        return None\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\n\ndef shmq_3level_gemm(\n    X: torch.Tensor,\n    W16: Optional[torch.Tensor],   # FP16 [N16, K]\n    W8: Optional[torch.Tensor],    # INT8 [N8, K]\n    W4: Optional[torch.Tensor],    # UINT8 [N4, K/2] (packed) OR INT8 [N4, K] (unpacked)\n    S8: Optional[torch.Tensor],    # FP16 [N8, K/128]\n    S4: Optional[torch.Tensor],    # FP16 [N4, K/128]\n    W4_packed: bool = True,        # True if W4 is uint8 packed (2 per byte)\n    require_cuda_kernel: bool = False,\n) -> torch.Tensor:\n    """3-level GEMM: Y = X @ [W16; W8; W4].T in a SINGLE kernel launch.\n\n    Args:\n        X: [M, K] or [M, *, K] FP16 on GPU. Multi-dim inputs are flattened to 2D.\n        W16: [N16, K] FP16 on GPU (or None if N16=0)\n        W8:  [N8, K]  INT8 on GPU (or None if N8=0)\n        W4:  [N4, K/2] UINT8 (packed) if W4_packed=True,\n             OR [N4, K] INT8 (unpacked) if W4_packed=False.\n             None if N4=0.\n        S8:  [N8, K/128] FP16 scales for INT8 (or None if N8=0)\n        S4:  [N4, K/128] FP16 scales for INT4 (or None if N4=0)\n        W4_packed: whether W4 is in packed uint8 form (default True).\n        require_cuda_kernel: raise instead of using the PyTorch fallback when\n            CuPy compilation or execution fails. Intended for CI/benchmarks.\n\n    Returns:\n        Y: [M, N] FP16 on GPU (or [M, *, N] reshaped to match X\'s leading dims).\n\n    Notes:\n        - If cupy is unavailable or kernel compilation fails, falls back to a\n          PyTorch reference implementation (correctness-only, ~10-50x slower).\n        - The fallback path uses the SAME math, so bit-exact equivalence is\n          expected (modulo FP32 reduction order).\n    """\n    # Flatten X to 2D\n    leading_shape = X.shape[:-1]\n    X2d = X.reshape(-1, X.shape[-1]).contiguous()\n    M, K = X2d.shape\n\n    N16 = W16.shape[0] if W16 is not None else 0\n    N8 = W8.shape[0] if W8 is not None else 0\n    if W4 is not None:\n        if W4_packed:\n            N4 = W4.shape[0]\n            assert W4.shape[1] == K // 2, \\\n                f"W4 packed shape {W4.shape} mismatch: expected [N4, {K//2}]"\n        else:\n            N4 = W4.shape[0]\n            assert W4.shape[1] == K, \\\n                f"W4 unpacked shape {W4.shape} mismatch: expected [N4, {K}]"\n    else:\n        N4 = 0\n    N = N16 + N8 + N4\n\n    # Validate K is divisible by GROUP_SIZE (128) — required for scale indexing\n    assert K % 128 == 0, \\\n        f"K ({K}) must be divisible by 128 (group_size) for scale indexing"\n\n    # Try cupy RawKernel path\n    cp = _check_cupy()\n    kernel = _get_gemm_kernel() if cp is not None else None\n\n    if kernel is not None and X2d.is_cuda:\n        try:\n            # If W4 is unpacked INT8, pack it to uint8 on GPU first\n            # (do this BEFORE the S3 contiguous check so we check the final tensor)\n            if W4 is not None and not W4_packed:\n                W4_eff = _pack_int4_on_gpu(W4)\n            else:\n                W4_eff = W4\n\n            # ---- Seam S3 hardening: log silent copies ----\n            # .contiguous() is called below on every tensor; if any of them\n            # is NOT already contiguous, this triggers a silent GPU memory\n            # copy. On 7B models this can spike memory by 3-5GB and OOM on T4.\n            # Log a warning (not an error) so we can diagnose in production.\n            _silent_copies = []\n            for _name, _t in [("X", X2d), ("W16", W16), ("W8", W8),\n                              ("W4", W4_eff), ("S8", S8), ("S4", S4)]:\n                if _t is not None and not _t.is_contiguous():\n                    _silent_copies.append(\n                        f"{_name}: shape={tuple(_t.shape)}, "\n                        f"strides={_t.stride()}, "\n                        f"size={_t.numel() * _t.element_size() // 1024}KB"\n                    )\n            if _silent_copies:\n                print(f"[shmq_3level_gemm] WARNING: silent contiguous copy on "\n                      f"{len(_silent_copies)} tensors (seam S3): "\n                      f"{\'; \'.join(_silent_copies[:3])}")\n\n            Y = torch.empty(M, N, dtype=torch.float16, device=X2d.device)\n\n            # Pack arguments — pass None as a 1-element dummy tensor of right type\n            def _to_cp(t, dtype, shape=(1,)):\n                if t is None:\n                    return cp.zeros(shape, dtype=dtype)\n                return cp.from_dlpack(t.detach().contiguous())\n\n            X_cp = cp.from_dlpack(X2d.detach().contiguous())\n            W16_cp = _to_cp(W16, cp.float16, (1, K))\n            W8_cp = _to_cp(W8, cp.int8, (1, K))\n            W4_cp = _to_cp(W4_eff, cp.uint8, (1, max(1, K // 2)))\n            S8_cp = _to_cp(S8, cp.float16, (1, max(1, K // 128)))\n            S4_cp = _to_cp(S4, cp.float16, (1, max(1, K // 128)))\n            Y_cp = cp.from_dlpack(Y)\n\n            # Grid: ceil(N/BN) x ceil(M/BM), block: 128 threads (4 warps)\n            BM, BN = 32, 32\n            grid = ((N + BN - 1) // BN, (M + BM - 1) // BM, 1)\n            block = (128, 1, 1)\n\n            # Shared memory: sX (32*32*2) + sW (32*32*2) = 4096 bytes\n            smem = 2 * 32 * 32 * 2\n\n            # DLPack does not synchronize PyTorch\'s non-default stream with\n            # CuPy. Launch on the producer stream so vLLM cannot observe a\n            # partially written output under asynchronous execution.\n            torch_stream = torch.cuda.current_stream(X2d.device)\n            with cp.cuda.ExternalStream(torch_stream.cuda_stream):\n                kernel(\n                    grid, block,\n                    (X_cp, W16_cp, W8_cp, W4_cp, S8_cp, S4_cp, Y_cp,\n                     M, K, N, N16, N8, N4),\n                    shared_mem=smem,\n                )\n            return Y.reshape(*leading_shape, N)\n        except Exception as e:\n            if require_cuda_kernel:\n                raise RuntimeError("SHMQ CUDA kernel execution failed") from e\n            print(f"[shmq_3level_gemm] cupy kernel failed ({e}); "\n                  f"falling back to PyTorch reference")\n\n    if require_cuda_kernel:\n        if not X2d.is_cuda:\n            raise RuntimeError("SHMQ CUDA kernel requires a CUDA input tensor")\n        if cp is None:\n            raise RuntimeError("SHMQ CUDA kernel required but CuPy is unavailable")\n        raise RuntimeError("SHMQ CUDA kernel required but NVRTC compilation failed")\n\n    # Fallback: PyTorch reference (correctness, slow)\n    return _pytorch_fallback(X2d, W16, W8, W4, S8, S4, W4_packed,\n                              N16, N8, N4, K, N).reshape(*leading_shape, N)\n\n\ndef _pack_int4_on_gpu(W4_int8: torch.Tensor) -> torch.Tensor:\n    """Pack int8 codes (values in [-8, 7]) into uint8 (2 per byte) on GPU.\n\n    Convention (MUST match the CUDA kernel `shmq_3level_gemm_kernel`):\n        LOW  nibble = EVEN index (gk even)\n        HIGH nibble = ODD  index (gk odd)\n    This is the same convention as MixLLM\'s `pack_int4_weights` and\n    `weight_packing.pack_int4`.\n    """\n    # W4_int8: [N4, K] int8, values in [-8, 7]\n    # Output:  [N4, K/2] uint8\n    N4, K = W4_int8.shape\n    # Take low 4 bits (two\'s-complement nibble for values in [-8, 7])\n    nibbles = (W4_int8.to(torch.int16) & 0x0F).to(torch.uint8)\n    low  = nibbles[:, 0::2]   # EVEN indices -> LOW nibble\n    high = nibbles[:, 1::2]   # ODD indices  -> HIGH nibble\n    packed = (high << 4) | low\n    return packed.contiguous()\n\n\ndef _pytorch_fallback(X, W16, W8, W4, S8, S4, W4_packed,\n                       N16, N8, N4, K, N) -> torch.Tensor:\n    """Pure-PyTorch reference implementation of the 3-level GEMM.\n\n    Computes Y = X @ [W16; W8; W4].T using FP32 accumulation, with explicit\n    dequantization of INT8/INT4 paths to FP16. Used for correctness checking\n    when cupy is unavailable, and as the verification oracle.\n    """\n    M = X.shape[0]\n    Xf = X.float()\n    Y = torch.zeros(M, N, dtype=torch.float16, device=X.device)\n\n    # FP16 path\n    if N16 > 0:\n        Y[:, :N16] = (Xf @ W16.float().T).to(torch.float16)\n\n    # INT8 path: dequant W8 with per-group scales, then matmul\n    if N8 > 0:\n        n_groups = K // 128\n        # W8: [N8, K] int8, S8: [N8, n_groups] fp16\n        W8_dq = (W8.float()\n                 .reshape(N8, n_groups, 128)\n                 * S8.float().unsqueeze(-1))   # broadcast [N8, n_groups, 1]\n        # The CUDA path stores dequantized tiles in FP16 shared memory.\n        W8_dq = W8_dq.to(torch.float16).float().reshape(N8, K)\n        Y[:, N16:N16 + N8] = (Xf @ W8_dq.T).to(torch.float16)\n\n    # INT4 path: unpack (if packed), dequant with per-group scales, matmul\n    if N4 > 0:\n        n_groups = K // 128\n        if W4_packed:\n            # W4: [N4, K/2] uint8 packed\n            # Convention (matches CUDA kernel + MixLLM + weight_packing.pack_int4):\n            #   LOW  nibble = EVEN index\n            #   HIGH nibble = ODD  index\n            low  = ( W4        & 0x0F).to(torch.int16)   # even indices\n            high = ((W4 >> 4) & 0x0F).to(torch.int16)   # odd indices\n            # Sign-extend from 4 bits: values 8..15 become -8..-1\n            low  = torch.where(low  >= 8, low  - 16, low)\n            high = torch.where(high >= 8, high - 16, high)\n            # Interleave: [low[0], high[0], low[1], high[1], ...]\n            W4_int8 = torch.stack([low, high], dim=-1).flatten(start_dim=1).to(torch.int8)\n        else:\n            W4_int8 = W4.to(torch.int8)\n        W4_dq = (W4_int8.float()\n                 .reshape(N4, n_groups, 128)\n                 * S4.float().unsqueeze(-1))\n        W4_dq = W4_dq.to(torch.float16).float().reshape(N4, K)\n        Y[:, N16 + N8:] = (Xf @ W4_dq.T).to(torch.float16)\n\n    return Y\n\n\nclass SHMQ3LevelKernel:\n    """High-level wrapper around the 3-level GEMM kernel.\n\n    Holds the 3 weight tensors in their native dtypes (FP16 / INT8 / packed\n    INT4) plus per-group scales. Used by `SHMQMixLLMLinear` (in\n    `mixllm/adapter.py`) as the compute backend.\n\n    Usage:\n        kern = SHMQ3LevelKernel.from_linear(linear_layer, n_bits_per_layer)\n        # kern.W16, kern.W8, kern.W4, kern.S8, kern.S4 are now populated\n        y = kern.forward(x)   # x: [..., K] FP16  ->  y: [..., N] FP16\n    """\n\n    def __init__(self,\n                 W16: Optional[torch.Tensor] = None,\n                 W8: Optional[torch.Tensor] = None,\n                 W4_packed: Optional[torch.Tensor] = None,\n                 S8: Optional[torch.Tensor] = None,\n                 S4: Optional[torch.Tensor] = None,\n                 K: int = 0, N: int = 0,\n                 N16: int = 0, N8: int = 0, N4: int = 0,\n                 group_size: int = 128):\n        if group_size != GROUP_SIZE:\n            raise ValueError(\n                f"SHMQ3LevelKernel supports group_size={GROUP_SIZE} only; "\n                f"got {group_size}. The CUDA kernel has fixed scale indexing."\n            )\n        self.W16 = W16\n        self.W8 = W8\n        self.W4 = W4_packed    # always stored as packed uint8\n        self.S8 = S8\n        self.S4 = S4\n        self.K = K\n        self.N = N\n        self.N16 = N16\n        self.N8 = N8\n        self.N4 = N4\n        self.group_size = group_size\n        cp = _check_cupy()\n        self.cupy_available = cp is not None\n        self.raw_kernel = _get_gemm_kernel() if self.cupy_available else None\n        if self.cupy_available and self.raw_kernel is None:\n            print("[SHMQ3LevelKernel] cupy available but kernel compile failed; "\n                  "falling back to PyTorch reference path")\n        if not self.cupy_available:\n            print("[SHMQ3LevelKernel] cupy not available; using PyTorch reference path "\n                  "(correctness-only, ~10-50x slower than CUDA)")\n\n    @property\n    def is_cuda_native(self) -> bool:\n        """True if the cupy.RawKernel is compiled and ready."""\n        return self.raw_kernel is not None\n\n    @property\n    def device(self) -> torch.device:\n        if self.W16 is not None:\n            return self.W16.device\n        if self.W8 is not None:\n            return self.W8.device\n        if self.W4 is not None:\n            return self.W4.device\n        return torch.device("cpu")\n\n    def to(self, device) -> "SHMQ3LevelKernel":\n        """Move all weight tensors to `device`."""\n        def _move(t):\n            return None if t is None else t.to(device)\n        return SHMQ3LevelKernel(\n            W16=_move(self.W16), W8=_move(self.W8), W4_packed=_move(self.W4),\n            S8=_move(self.S8), S4=_move(self.S4),\n            K=self.K, N=self.N, N16=self.N16, N8=self.N8, N4=self.N4,\n            group_size=self.group_size,\n        )\n\n    def forward(self, X: torch.Tensor) -> torch.Tensor:\n        """Compute Y = X @ [W16; W8; W4].T as a single-kernel GEMM.\n\n        Args:\n            X: [..., K] FP16 (any number of leading dims)\n\n        Returns:\n            Y: [..., N] FP16 (same leading dims as X)\n        """\n        # Per-token activation quantization is NOT applied here — we keep\n        # activations FP16 to preserve the value of the FP16 weight path.\n        # (Original SHMQ paper uses W4.8A8, but for the 3-level {4,8,16}\n        # design, FP16 activations are the natural choice.)\n        return shmq_3level_gemm(\n            X, self.W16, self.W8, self.W4, self.S8, self.S4,\n            W4_packed=True,\n        )\n\n    @staticmethod\n    def from_weight_pack(pack: Dict[str, torch.Tensor],\n                          n_bits: int,\n                          device: Optional[torch.device] = None,\n                          group_size: int = 128) -> "SHMQ3LevelKernel":\n        """Build a SHMQ3LevelKernel from a pack dict (output of\n        `inference.weight_packing.pack_shmq_linear`).\n\n        For 2-level packs ({4, 8} only — original SHMQ), this places:\n          - INT8 portion into W8\n          - INT4 portion into W4\n          - W16 = None\n        For full 3-level packs ({4, 8, 16}), the caller should provide\n        separate W16, W8, W4 buffers and use the constructor directly.\n\n        Args:\n            pack: dict with keys qweight_int8, scales_int8, qweight_int4,\n                  scales_int4, in_features, out_features, n_sensitive.\n            n_bits: the per-layer bit-width (4 or 8) — used to decide\n                    whether to use W8 (8-bit) or W4 (4-bit) exclusively.\n        """\n        K = pack["in_features"]\n        N = pack["out_features"]\n        n_sens = pack["n_sensitive"]\n\n        W8 = pack.get("qweight_int8", None)\n        S8 = pack.get("scales_int8", None)\n        W4 = pack.get("qweight_int4", None)\n        S4 = pack.get("scales_int4", None)\n\n        if device is not None:\n            W8 = W8.to(device) if W8 is not None else None\n            S8 = S8.to(device) if S8 is not None else None\n            W4 = W4.to(device) if W4 is not None else None\n            S4 = S4.to(device) if S4 is not None else None\n\n        # For 2-level SHMQ packs:\n        #   if n_bits == 8: layer is fully 8-bit (W4=None, W8 has N rows)\n        #   if n_bits == 4: layer is mixed (W8 has n_sens rows, W4 has N-n_sens rows)\n        if n_bits == 8:\n            N8 = N\n            N4 = 0\n            N16 = 0\n        else:\n            N8 = n_sens\n            N4 = N - n_sens\n            N16 = 0\n\n        return SHMQ3LevelKernel(\n            W16=None, W8=W8, W4_packed=W4, S8=S8, S4=S4,\n            K=K, N=N, N16=N16, N8=N8, N4=N4,\n            group_size=group_size,\n        )\n\n\n# ---------------------------------------------------------------------------\n# Smoke test / verification helpers\n# ---------------------------------------------------------------------------\n\ndef verify_against_pytorch(M: int = 64, K: int = 256, N: int = 96,\n                            N16: int = 32, N8: int = 32, N4: int = 32,\n                            device: str = "cuda",\n                            tol: float = 1e-2) -> Tuple[torch.Tensor, torch.Tensor, float]:\n    """Generate random inputs, run both cupy kernel and PyTorch fallback,\n    return (Y_cuda, Y_ref, max_abs_diff).\n\n    Useful as a sanity check after kernel compilation. Requires cupy + GPU.\n    """\n    torch.manual_seed(42)\n    X = torch.randn(M, K, dtype=torch.float16, device=device) * 0.1\n    W16 = torch.randn(N16, K, dtype=torch.float16, device=device) * 0.1\n    W8 = torch.randint(-127, 127, (N8, K), dtype=torch.int8, device=device)\n    W4_codes = torch.randint(-7, 8, (N4, K), dtype=torch.int8, device=device)\n    # Pack W4\n    W4_packed = _pack_int4_on_gpu(W4_codes)\n    n_groups = K // 128\n    S8 = torch.randn(N8, n_groups, dtype=torch.float16, device=device) * 0.01\n    S4 = torch.randn(N4, n_groups, dtype=torch.float16, device=device) * 0.1\n\n    # CUDA path\n    Y_cuda = shmq_3level_gemm(\n        X, W16, W8, W4_packed, S8, S4,\n        W4_packed=True,\n        require_cuda_kernel=True,\n    )\n\n    # Reference path (force PyTorch fallback by temporarily disabling cupy)\n    global _CUPY_AVAILABLE\n    saved = _CUPY_AVAILABLE\n    _CUPY_AVAILABLE = False\n    try:\n        Y_ref = shmq_3level_gemm(X, W16, W8, W4_packed, S8, S4, W4_packed=True)\n    finally:\n        _CUPY_AVAILABLE = saved\n\n    diff = (Y_cuda.float() - Y_ref.float()).abs().max().item()\n    return Y_cuda, Y_ref, diff\n'
module = type(sys)('shmq_kernel_under_test')
module.__file__ = 'embedded:shmq_3level_kernel.py'
sys.modules[module.__name__] = module
exec(compile(_source, module.__file__, 'exec'), module.__dict__)
print('Loaded kernel source:', len(module.SHMQ_3LEVEL_KERNEL_CUDA), 'CUDA chars')
assert 'mma.sync' not in module.SHMQ_3LEVEL_KERNEL_CUDA
print('Confirmed: this gate measures the implemented CUDA-cores path, not an unverified MMA claim.')


In [ ]:
# Strict correctness tests. Every call requires the actual RawKernel path.
torch.manual_seed(1234)
cases = [(1, 128, 1, 1, 1), (7, 256, 17, 19, 23), (33, 384, 32, 32, 32), (64, 512, 31, 37, 29)]
results = []
for M, K, N16, N8, N4 in cases:
    device = 'cuda'
    X = torch.randn(M, K, dtype=torch.float16, device=device) * 0.1
    W16 = torch.randn(N16, K, dtype=torch.float16, device=device) * 0.1
    W8 = torch.randint(-127, 128, (N8, K), dtype=torch.int8, device=device)
    W4_codes = torch.randint(-8, 8, (N4, K), dtype=torch.int8, device=device)
    W4 = module._pack_int4_on_gpu(W4_codes)
    groups = K // 128
    S8 = torch.rand(N8, groups, dtype=torch.float16, device=device) * 0.02
    S4 = torch.rand(N4, groups, dtype=torch.float16, device=device) * 0.1
    t0 = time.perf_counter()
    actual = module.shmq_3level_gemm(X, W16, W8, W4, S8, S4, require_cuda_kernel=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    reference = module._pytorch_fallback(X, W16, W8, W4, S8, S4, True, N16, N8, N4, K, N16 + N8 + N4)
    diff = (actual.float() - reference.float()).abs().max().item()
    results.append((M, K, N16, N8, N4, diff, elapsed))
    print(f'M={M:3d} K={K:3d} regions=({N16},{N8},{N4}) max_abs_diff={diff:.6g} elapsed={elapsed:.3f}s')
    assert diff <= 2e-2, f'kernel mismatch: {diff}'
print('STRICT CUDA CORRECTNESS: PASS')


In [ ]:
# Test the public verification helper as an additional no-fallback check.
_, _, helper_diff = module.verify_against_pytorch(M=64, K=256, N=96, N16=32, N8=32, N4=32)
print('verify_against_pytorch max_abs_diff:', helper_diff)
assert helper_diff <= 2e-2
print('GPU GATE: PASS')
